## 1. 研究目标（层次聚类轨迹）

对每个 **(国家, 主题)**，在多年份上观察相对研究前沿的强度（`ratio_to_frontier`），经 **时间滚动平滑** 后得到一条一维时间序列。本 notebook 使用 **Hierarchical Clustering（层次聚类）** 对这些轨迹聚类，距离模式由 `HIER_DISTANCE_MODE` 控制：

- **euclidean**：列向标准化后的欧氏距离（特征空间 linkage；可与 `ward` / `average` / `complete` / `single` 搭配）。
- **dtw**：两两 **DTW** 预计算距离矩阵 + linkage（推荐 **average**；需要 `tslearn`）。
- **shape**：行内 **z-normalization** 后，用 **correlation distance** \(1 - \mathrm{corr}(x,y)\) 作为形状距离（推荐 **average**）。

默认经验组合：**dtw + average**、**shape + average**、**euclidean + ward 或 average**。

**输出目录**：主目录为 `output/time_series_hierarchical/`（`trajectory_*_hier_<mode>.csv/png` 等）。**不覆盖** `output/catchup_mvp/topic_country_year_panel.csv`；本 notebook 另写出 **`trajectory_units_dimred_2d_hier_<mode>.csv`**（每行一个 (country, topic) + UMAP/t-SNE/PCA 二维坐标与聚类标签），以及 **`topic_country_year_panel_with_traj_embeddings_hier_<mode>.csv`**（原面板列左连接同上嵌入与 `trajectory_cluster`）。亦**不覆盖** `country_topic_catchup_dtw.ipynb` 对应的 `*_dtw.*` 文件。


## 2. 数据依赖

- **配置**：[`config/cluster_keybert_from_cluster.json`](config/cluster_keybert_from_cluster.json)（数据 CSV 路径、`paper_topics.csv` 所在目录、rolling 窗口与随机种子）。
- **面板**：优先读取 `output/catchup_mvp/topic_country_year_panel.csv`（与 MVP 一致）；若不存在且 frontier 为 `top3_mean`，则从 `paper_topics.csv` 或配置中的主 CSV 重建；若仍缺文件会报错并提示先运行上游 notebook。

**内核**：建议使用仓库 `.venv`（含 `umap-learn`；**dtw** 模式另需 `tslearn`）。在 Jupyter 中选择与项目一致的内核，或执行：  
`uv run python -m ipykernel install --user --name=catchup-venv --display-name='catch-up (uv venv)'` 后刷新内核列表。

§5 树状图使用 **scipy**（`linkage` / `dendrogram`）。


In [11]:
from __future__ import annotations

import json
import os
import random
import warnings
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# 屏蔽部分库的未来弃用警告，避免输出干扰阅读（不影响数值结果）。
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 路径与输出目录（相对仓库根目录）---
_CONFIG_PATH = Path("config/cluster_keybert_from_cluster.json")
CATCHUP_MVP_DIR = Path("output/catchup_mvp")
CATCHUP_ROUND2_DIR = Path("output/time_series_hierarchical")
CATCHUP_ROUND2_DIR.mkdir(parents=True, exist_ok=True)


def load_json_config(path: Path) -> dict[str, Any]:
    """读取聚类流水线 JSON 配置。"""
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def set_reproducibility(seed: int) -> None:
    """固定 Python / NumPy 随机种子，便于结果复现。"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


_cfg = load_json_config(_CONFIG_PATH)
_paths = _cfg["paths"]
DATA_PATH = Path(_paths["data_csv"])
CLUSTER_OUTPUT_DIR = Path(_paths["output_dir"])
PAPER_TOPICS_PATH = CLUSTER_OUTPUT_DIR / "paper_topics.csv"
_time = _cfg.get("time_evolution", {})
CONFIG_ROLLING_WINDOW = int(_time.get("rolling_window_size", 5))
CONFIG_ROLLING_MIN = int(_time.get("rolling_min_periods", 3))
RANDOM_STATE = int(_cfg["reproducibility"]["seed"])
set_reproducibility(RANDOM_STATE)

print("CATCHUP_MVP_DIR exists:", CATCHUP_MVP_DIR.is_dir())
print("CATCHUP_ROUND2_DIR:", CATCHUP_ROUND2_DIR.resolve())


CATCHUP_MVP_DIR exists: True
CATCHUP_ROUND2_DIR: /Users/luoyiti/Project/catch-up/output/time_series_hierarchical


## 3. 参数区（集中管理）

In [12]:
# 与配置中的随机种子一致（上一单元已从 JSON 写入 RANDOM_STATE）。
RANDOM_STATE = RANDOM_STATE

# 轨迹样本至少需要在多少年份上有发文，才进入聚类（避免极稀疏噪声单元）。
MIN_ACTIVE_YEARS = 6

# 对 ratio_to_frontier 做滚动平均的窗口长度与最小有效窗口（与 MVP / KeyBERT 配置对齐）。
MAIN_ROLLING_WINDOW = CONFIG_ROLLING_WINDOW
ROLLING_MIN_PERIODS = CONFIG_ROLLING_MIN

# 前沿强度定义：在每个 (topic, year) 上取发文量 Top-K 国家的平均 share 作为 frontier（top3_mean）。
FRONTIER_TOP_K = 3
MAIN_FRONTIER_MODE: Literal["top3_mean", "max"] = "top3_mean"

# 层次聚类：距离模式与 linkage；在 k 的网格上搜索（dtw/shape 的预计算距离为 O(n²)，上界过大将显著变慢）。
HIER_DISTANCE_MODE: Literal["euclidean", "dtw", "shape"] = "dtw"
HIER_LINKAGE_METHOD: Literal["ward", "average", "complete", "single"] = "average"
TRAJ_K_RANGE = range(2, 11)

# 最终聚类数：metrics_lex = 在 X_score 上 (DB 升序, Silhouette 降序) 字典序最优；fixed = 使用 FINAL_K_OVERRIDE。
FINAL_K_STRATEGY: Literal["metrics_lex", "fixed"] = "metrics_lex"
FINAL_K_OVERRIDE: int | None = 3  # only when FINAL_K_STRATEGY == "fixed"

# 设为非空列表（如 [3, 5, 7]）时，在 §5 末尾用各窗口重建轨迹并以当前 k_final 各拟合一次，输出对比小表（默认关闭）。
SWEEP_ROLLING_WINDOWS: list[int] = []

# 过滤弱信号单元：全时段内「单年最大主题份额」低于该阈值的 (country, topic) 剔除（减少长期接近 0 的轨迹）。
MIN_PEAK_TOPIC_YEAR_SHARE = 0.05
# 若 >0，则还要求「有发文年份上的平均份额」不低于该阈值；0 表示关闭此约束。
MIN_MEAN_SHARE_ACTIVE = 0.0

# UMAP / t-SNE / PCA 仅用于二维可视化展平后的 rolling 特征（或与 shape 评分一致的 z-score 矩阵），不参与聚类。
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.1
# t-SNE perplexity 会在运行时夹紧到 [5, n_samples-1]。
TSNE_PERPLEXITY = 30

# ratio_to_frontier = share / frontier 的分母下界：避免 frontier 为极小正数时比值爆炸为 inf（不修改 share）。
FRONTIER_RATIO_EPS = 1e-10
# 透视与 rolling 之后将非有限值压到该上界，防止历史 CSV 中残留 inf 进入距离计算。
RATIO_ROLLING_MAX_CLIP = 50.0


def validate_hierarchical_params(distance_mode: str, linkage_method: str) -> None:
    """Validate HIER_DISTANCE_MODE x HIER_LINKAGE_METHOD combinations."""
    if distance_mode not in ("euclidean", "dtw", "shape"):
        raise ValueError(f"Invalid HIER_DISTANCE_MODE: {distance_mode!r}")
    if linkage_method not in ("ward", "average", "complete", "single"):
        raise ValueError(f"Invalid HIER_LINKAGE_METHOD: {linkage_method!r}")
    if distance_mode in ("dtw", "shape") and linkage_method == "ward":
        raise ValueError("ward linkage is not valid with precomputed DTW or shape distances; use average/complete/single.")


validate_hierarchical_params(HIER_DISTANCE_MODE, HIER_LINKAGE_METHOD)

plt.rcParams["figure.dpi"] = 120
print(
    "主要参数:",
    "MAIN_ROLLING_WINDOW=",
    MAIN_ROLLING_WINDOW,
    "MAIN_FRONTIER_MODE=",
    MAIN_FRONTIER_MODE,
    "HIER_DISTANCE_MODE=",
    HIER_DISTANCE_MODE,
    "HIER_LINKAGE_METHOD=",
    HIER_LINKAGE_METHOD,
    "TRAJ_K_RANGE=",
    list(TRAJ_K_RANGE),
    "FINAL_K_STRATEGY=",
    FINAL_K_STRATEGY,
    "FINAL_K_OVERRIDE=",
    FINAL_K_OVERRIDE,
    "SWEEP_ROLLING_WINDOWS=",
    SWEEP_ROLLING_WINDOWS,
    "MIN_PEAK_TOPIC_YEAR_SHARE=",
    MIN_PEAK_TOPIC_YEAR_SHARE,
    "FRONTIER_RATIO_EPS=",
    FRONTIER_RATIO_EPS,
    "RATIO_ROLLING_MAX_CLIP=",
    RATIO_ROLLING_MAX_CLIP,
    "TSNE_PERPLEXITY=",
    TSNE_PERPLEXITY,
)


主要参数: MAIN_ROLLING_WINDOW= 5 MAIN_FRONTIER_MODE= top3_mean HIER_DISTANCE_MODE= dtw HIER_LINKAGE_METHOD= average TRAJ_K_RANGE= [2, 3, 4, 5, 6, 7, 8, 9, 10] FINAL_K_STRATEGY= metrics_lex FINAL_K_OVERRIDE= 3 SWEEP_ROLLING_WINDOWS= [] MIN_PEAK_TOPIC_YEAR_SHARE= 0.05 FRONTIER_RATIO_EPS= 1e-10 RATIO_ROLLING_MAX_CLIP= 50.0 TSNE_PERPLEXITY= 30


## 4. 轨迹矩阵构建

从面板得到每个 **(country, topic, year)** 的 `ratio_to_frontier`（相对前沿比值），再按年透视、对缺失年填 0 后做 **rolling 平滑**，得到 `wide_roll_raw`：形状为 **(样本数, 年份数)**，每一行是一条待聚类的时间序列。随后用 `MIN_ACTIVE_YEARS` 与峰值份额阈值筛掉不可靠单元。

**数值稳定**：`ratio = share / max(frontier_value, FRONTIER_RATIO_EPS)`，不改动 `share` 单纯形；读入已有 MVP 面板 CSV 时若含 `share` 与 `frontier_value` 会用同一公式重算 `ratio_to_frontier`。透视与 rolling 后对轨迹做有限幅裁剪，防止历史数据中的 `inf` 进入后续距离计算。


In [13]:
# 列名候选：兼容 WoS 导出或中间表的不同命名。
TOPIC_CANDIDATES = ("topic", "Topic", "topic_id")
YEAR_CANDIDATES = ("year", "Year", "pub_year", "publication_year")
COUNTRY_CANDIDATES = ("country_code", "country", "Country", "nation")


def _first_present(df: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    for n in names:
        if n in df.columns:
            return n
    return None


def resolve_topic_country_year_columns(df: pd.DataFrame) -> dict[str, str]:
    out: dict[str, str] = {}
    out["topic"] = _first_present(df, TOPIC_CANDIDATES) or ""
    out["year"] = _first_present(df, YEAR_CANDIDATES) or ""
    out["country_src"] = _first_present(df, COUNTRY_CANDIDATES) or ""
    missing = [k for k, v in out.items() if not v]
    if missing:
        raise KeyError(f"Missing columns for: {missing}. Available: {list(df.columns)[:40]}...")
    return out  # type: ignore[return-value]


def load_paper_level_table() -> tuple[pd.DataFrame, str]:
    """优先读聚类输出的 paper_topics；否则读配置中的主 CSV。"""
    if PAPER_TOPICS_PATH.exists():
        df = pd.read_csv(PAPER_TOPICS_PATH, low_memory=False)
        return df, str(PAPER_TOPICS_PATH)
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Neither {PAPER_TOPICS_PATH} nor {DATA_PATH} exists. "
            "Run cluster_keybert_from_cluster.ipynb to export paper_topics.csv."
        )
    df = pd.read_csv(DATA_PATH, low_memory=False)
    if _first_present(df, TOPIC_CANDIDATES) is None:
        raise ValueError(f"No topic column in {DATA_PATH}.")
    return df, str(DATA_PATH)


def frontier_topk_mean_share(shares: pd.Series, k: int) -> float:
    """在每个 (topic, year) 内，对各国 share 取 top-k 正的国家的均值，作为前沿参考。"""
    s = shares.astype(float).sort_values(ascending=False)
    s = s[s > 0] if (s > 0).any() else s
    if s.empty:
        return 0.0
    top = s.head(min(k, len(s)))
    return float(top.mean())


def frontier_max_share(shares: pd.Series) -> float:
    """备选前沿：该国在该主题该年的最大 share（更激进）。"""
    s = shares.astype(float)
    if s.empty:
        return 0.0
    return float(s.max())


def build_topic_country_year_panel_with_frontier(
    papers: pd.DataFrame,
    frontier_mode: Literal["top3_mean", "max"],
    top_k: int,
) -> pd.DataFrame:
    """构造完整 (topic, country, year) 网格，计算 count、share、rank、frontier_value、ratio_to_frontier。"""
    counts = (
        papers.groupby(["topic", "country", "year"], observed=True)
        .size()
        .rename("count")
        .reset_index()
    )
    topics = counts["topic"].unique()
    countries = counts["country"].unique()
    years = np.arange(int(counts["year"].min()), int(counts["year"].max()) + 1, dtype=int)
    full = pd.MultiIndex.from_product(
        [topics, countries, years], names=["topic", "country", "year"]
    ).to_frame(index=False)
    panel = full.merge(counts, on=["topic", "country", "year"], how="left")
    panel["count"] = panel["count"].fillna(0).astype(int)
    tot = panel.groupby(["topic", "year"])["count"].transform("sum")
    panel["share"] = np.where(tot > 0, panel["count"] / tot, 0.0)
    panel["rank"] = panel.groupby(["topic", "year"])["count"].rank(ascending=False, method="min")
    if frontier_mode == "top3_mean":
        panel["frontier_value"] = panel.groupby(["topic", "year"])["share"].transform(
            lambda s: frontier_topk_mean_share(s, top_k)
        )
    elif frontier_mode == "max":
        panel["frontier_value"] = panel.groupby(["topic", "year"])["share"].transform(frontier_max_share)
    else:
        raise ValueError(frontier_mode)
    # 安全除法：分母不低于 FRONTIER_RATIO_EPS，避免极小 frontier_value 导致 inf（share 不加扰动）。
    fv = panel["frontier_value"].astype(float)
    sh = panel["share"].astype(float)
    denom = np.maximum(fv, FRONTIER_RATIO_EPS)
    panel["ratio_to_frontier"] = (sh / denom).astype(float)
    panel["gap_to_frontier"] = panel["frontier_value"] - panel["share"]
    return panel


def load_or_build_panel(frontier_mode: str, top_k: int) -> tuple[pd.DataFrame, str]:
    """若 MVP 已生成面板且模式为 top3_mean，则直接读取；否则从论文表重建。"""
    mvp_panel = CATCHUP_MVP_DIR / "topic_country_year_panel.csv"
    if mvp_panel.exists() and frontier_mode == "top3_mean":
        panel = pd.read_csv(mvp_panel, low_memory=False)
        # 与重建路径一致：用分母下界重算 ratio，修复旧 CSV 中因极小 frontier 产生的 inf。
        if {"share", "frontier_value"}.issubset(panel.columns):
            fv = panel["frontier_value"].astype(float)
            sh = panel["share"].astype(float)
            panel["ratio_to_frontier"] = (sh / np.maximum(fv, FRONTIER_RATIO_EPS)).astype(float)
        src = f"read:{mvp_panel}"
    else:
        papers, _src = load_paper_level_table()
        cols = resolve_topic_country_year_columns(papers)
        tc, yc, csc = cols["topic"], cols["year"], cols["country_src"]
        if csc != "country" and "country" in papers.columns:
            papers = papers.drop(columns=["country"], errors="ignore")
        papers = papers.rename(columns={tc: "topic", yc: "year", csc: "country"})
        papers["topic"] = pd.to_numeric(papers["topic"], errors="coerce")
        papers["year"] = pd.to_numeric(papers["year"], errors="coerce").astype("Int64")
        papers = papers.dropna(subset=["topic", "year", "country"]).copy()
        papers["topic"] = papers["topic"].astype(int)
        papers["year"] = papers["year"].astype(int)
        papers = papers[papers["topic"] != -1]
        papers["country"] = papers["country"].astype(str).str.strip()
        papers = papers[papers["country"].str.len() > 0]
        panel = build_topic_country_year_panel_with_frontier(papers, frontier_mode, top_k)  # type: ignore[arg-type]
        src = f"rebuild:{frontier_mode}"
        if not mvp_panel.exists():
            out_rebuilt = CATCHUP_ROUND2_DIR / "topic_country_year_panel_rebuilt.csv"
            panel.to_csv(out_rebuilt, index=False)
            print("Saved rebuilt panel to", out_rebuilt)
    return panel, src


panel_main, panel_src = load_or_build_panel(MAIN_FRONTIER_MODE, FRONTIER_TOP_K)
print("Panel source:", panel_src)
print("shape", panel_main.shape, "cols", list(panel_main.columns))
print(panel_main.isna().mean().sort_values(ascending=False).head(8))
print(panel_main.head(3))


def build_trajectory_matrix(
    panel: pd.DataFrame,
    years_all: np.ndarray,
    rolling_window: int,
    min_periods: int,
    min_active_years: int,
    min_peak_share: float | None = None,
    min_mean_share_active: float | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.DataFrame]:
    """由面板得到 rolling 后的 ratio 轨迹及辅助宽表；返回的 wide_roll 行索引为 MultiIndex (country, topic)。"""
    year_cols = [str(y) for y in years_all]
    thr = MIN_PEAK_TOPIC_YEAR_SHARE if min_peak_share is None else float(min_peak_share)
    mthr = MIN_MEAN_SHARE_ACTIVE if min_mean_share_active is None else float(min_mean_share_active)
    ratio_pivot = panel.pivot_table(
        index=["country", "topic"], columns="year", values="ratio_to_frontier", aggfunc="first"
    )
    ratio_pivot = ratio_pivot.reindex(columns=years_all)
    ratio_filled = ratio_pivot.fillna(0.0)
    # 将 nan/inf 压到有限区间，避免读入历史面板时残留 inf 进入 rolling。
    ratio_filled = pd.DataFrame(
        np.nan_to_num(
            ratio_filled.to_numpy(dtype=float, copy=True),
            nan=0.0,
            posinf=RATIO_ROLLING_MAX_CLIP,
            neginf=0.0,
        ),
        index=ratio_filled.index,
        columns=ratio_filled.columns,
    )
    # 按时间轴 rolling：平滑单年噪声，使 DTW 更关注趋势与阶段而非单点毛刺。
    roll = ratio_filled.T.rolling(rolling_window, min_periods=min_periods).mean().T.fillna(0.0)
    roll = pd.DataFrame(
        np.nan_to_num(
            roll.to_numpy(dtype=float, copy=True),
            nan=0.0,
            posinf=RATIO_ROLLING_MAX_CLIP,
            neginf=0.0,
        ),
        index=roll.index,
        columns=roll.columns,
    )
    count_pivot = (
        panel.pivot_table(index=["country", "topic"], columns="year", values="count", aggfunc="first")
        .reindex(columns=years_all)
        .fillna(0)
        .astype(int)
    )
    share_pivot = panel.pivot_table(
        index=["country", "topic"], columns="year", values="share", aggfunc="first"
    ).reindex(columns=years_all)
    peak_share = share_pivot.max(axis=1).fillna(0.0)
    mask_peak = peak_share >= thr
    active_years = (count_pivot > 0).sum(axis=1)
    mask_active = active_years >= min_active_years
    ps_active = peak_share.reindex(active_years.index).fillna(0.0)
    print("peak_share among active_years-eligible units:\n", ps_active.loc[mask_active].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    count_pos = count_pivot > 0
    mean_share_active = share_pivot.where(count_pos).mean(axis=1).fillna(0.0)
    if mthr > 0:
        mean_ok = mean_share_active.reindex(active_years.index).fillna(0.0) >= mthr
        print("mean_share_active (count>0 years) among active-eligible:\n", mean_share_active.reindex(active_years.index).fillna(0.0).loc[mask_active].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    else:
        mean_ok = pd.Series(True, index=active_years.index)
    peak_ok = mask_peak.reindex(active_years.index).fillna(False)
    mask = mask_active & peak_ok & mean_ok
    n_active = int(mask_active.sum())
    n_final = int(mask.sum())
    n_drop_peak = int((mask_active & ~peak_ok).sum())
    n_drop_mean = int((mask_active & peak_ok & ~mean_ok).sum()) if mthr > 0 else 0
    print(
        f"Trajectory filter: active_years>={min_active_years}: {n_active}, "
        f"after peak_share>={thr} and mean_share_active>={mthr}: {n_final}, "
        f"dropped_by_peak_only: {n_drop_peak}, dropped_by_mean_only: {n_drop_mean}"
    )
    wide_roll = roll.loc[mask].copy()
    wide_roll.columns = year_cols
    wide_ratio = ratio_filled.loc[mask].copy()
    wide_ratio.columns = year_cols
    active_f = active_years.loc[mask]
    count_sub = count_pivot.loc[mask].copy()
    count_sub.columns = year_cols
    return wide_roll, wide_ratio, active_f, count_sub


def z_normalize_rows(wide_roll: pd.DataFrame) -> np.ndarray:
    """Per-row z-score; constant or invalid rows become zeros (finite, safe for correlation)."""
    X = wide_roll.to_numpy(dtype=float, copy=True)
    mu = np.nanmean(X, axis=1, keepdims=True)
    sig = np.nanstd(X, axis=1, ddof=0, keepdims=True)
    bad = ~np.isfinite(sig) | (sig < 1e-12)
    Z = (X - mu) / np.where(bad, 1.0, sig)
    Z = np.where(bad, 0.0, Z)
    Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
    return Z


years_all = np.sort(panel_main["year"].unique())
# wide_roll_raw：层次聚类输入使用的原始滚动水平（不做行内 z-score，以保留与前沿的绝对距离信息；shape 模式在 §5 另行 z-normalize）。
wide_roll_raw, wide_ratio_raw, active_years_s, count_wide = build_trajectory_matrix(
    panel_main, years_all, MAIN_ROLLING_WINDOW, ROLLING_MIN_PERIODS, MIN_ACTIVE_YEARS
)
meta = wide_roll_raw.reset_index()[["country", "topic"]]
print("Trajectory units:", len(wide_roll_raw), "T:", wide_roll_raw.shape[1], "years", years_all[0], "..", years_all[-1])


Panel source: read:output/catchup_mvp/topic_country_year_panel.csv
shape (98050, 9) cols ['topic', 'country', 'year', 'count', 'share', 'rank', 'frontier_value', 'ratio_to_frontier', 'gap_to_frontier']
topic                0.0
country              0.0
year                 0.0
count                0.0
share                0.0
rank                 0.0
frontier_value       0.0
ratio_to_frontier    0.0
dtype: float64
   topic    country  year  count  share  rank  frontier_value  \
0      0  Argentina  1990      0    0.0   4.0        0.333333   
1      0  Argentina  1991      0    0.0   6.0        0.285714   
2      0  Argentina  1992      0    0.0   9.0        0.200000   

   ratio_to_frontier  gap_to_frontier  
0                0.0         0.333333  
1                0.0         0.285714  
2                0.0         0.200000  
peak_share among active_years-eligible units:
 count    670.000000
mean       0.420326
std        0.298068
min        0.032258
10%        0.100000
25%        0.20

## 5. 层次聚类（Hierarchical Clustering）与导出

- **模型**：`sklearn.cluster.AgglomerativeClustering`；`euclidean` 在标准化特征上直接拟合；`dtw` / `shape` 使用 **metric="precomputed"** 与两两距离矩阵。
- **全 k 诊断**：在 `TRAJ_K_RANGE` 内对每个 k 拟合一次，在二维特征矩阵 `X_score` 上计算 **Davies–Bouldin** 与 **轮廓系数**（`silhouette` 失败则记为 NaN），保存 `trajectory_k_search_metrics_hier_<mode>.csv`、绘制 `trajectory_k_diagnostics_hier_<mode>.png`（仅 k–DB 与 k–Silhouette，并标出 `k_final`）。
- **最终 k**：`FINAL_K_STRATEGY` 为 `metrics_lex`（先最小化 DB，再最大化 silhouette）或 `fixed`（`FINAL_K_OVERRIDE`）；**不再**使用 inertia 肘部。
- **树状图**：`scipy.cluster.hierarchy` + 大样本时可截断显示；导出 `trajectory_dendrogram_hier_<mode>.png`。
- **产出**：`trajectory_clusters_hier_<mode>.csv`、`trajectory_units_dimred_2d_hier_<mode>.csv`（每行一个 **(country, topic)** 轨迹单元：`trajectory_cluster` + `umap_1/2`、`tsne_1/2`、`pca_1/2` 及 `tsne_perplexity_used`、`pca_pc1_pc2_variance_explained_pct` 等）、`trajectory_prototypes_hier_<mode>.png`、以及二维嵌入可视化 `trajectory_umap_hier_<mode>.png`、`trajectory_tsne_hier_<mode>.png`、`trajectory_pca_hier_<mode>.png`（特征与 UMAP 一致：rolling 列标准化；shape 模式为行 z-score）。同时将 `trajectory_cluster`、`traj_umap_*` / `traj_tsne_*` / `traj_pca_*` 及 `traj_embed_*` 元数据列 **左连接** 到与 §4 相同的 `panel_main`（与 `output/catchup_mvp/topic_country_year_panel.csv` 同 schema 的行表），写出 `output/time_series_hierarchical/topic_country_year_panel_with_traj_embeddings_hier_<mode>.csv`，并在 `output/catchup_mvp/` 下保存同名副本（**不覆盖**原 `topic_country_year_panel.csv`）。**dtw** 模式下两两 DTW 为 **O(n²)** 时间与内存，样本量大时慎用上界较大的 `TRAJ_K_RANGE`。


In [14]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

def plot_trajectory_prototypes(
    wide_roll: pd.DataFrame,
    labels: np.ndarray,
    years: np.ndarray,
    out_path: Path,
    title: str,
) -> None:
    import warnings
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    """Mean rolling ratio_to_frontier curves per cluster (prototype interpretation)."""
    # 只选择 year >= 1992 的年份
    mask_year = years >= 1992
    years_filtered = years[mask_year]
    wide_roll_filtered = wide_roll.iloc[:, mask_year]

    clusters = np.sort(np.unique(labels))
    fig, ax = plt.subplots(figsize=(10, 5))
    for c in clusters:
        idx = labels == c
        mean_curve = wide_roll_filtered.values[idx].mean(axis=0)
        ax.plot(years_filtered, mean_curve, label=f"C{int(c)} (n={int(idx.sum())})")
    ax.axhline(1.0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Year")
    ax.set_ylabel("Mean rolling ratio_to_frontier")
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def _viz_feature_matrix(
    wide_roll: pd.DataFrame,
    X_umap_feat: np.ndarray | None,
) -> np.ndarray:
    """Same input space as UMAP: column-standardized rolling, or precomputed rows (shape z-score)."""
    if X_umap_feat is None:
        return StandardScaler().fit_transform(wide_roll.values.astype(float))
    return np.asarray(X_umap_feat, dtype=float)


def fit_trajectory_2d_embeddings(
    wide_roll: pd.DataFrame,
    X_umap_feat: np.ndarray | None,
    random_state: int,
) -> dict[str, Any]:
    """
    One-shot UMAP / t-SNE / PCA on visualization features; row-aligned with wide_roll.
    Returns arrays (n, 2); umap/tsne are nan when n < 5, pca when n < 3.
    """
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from umap import UMAP

    X = _viz_feature_matrix(wide_roll, X_umap_feat)
    n = X.shape[0]
    out_umap = np.full((n, 2), np.nan, dtype=float)
    out_tsne = np.full((n, 2), np.nan, dtype=float)
    out_pca = np.full((n, 2), np.nan, dtype=float)
    tsne_perp: int | None = None
    pca_var_pct: float | None = None
    if n >= 3:
        pca = PCA(n_components=2)
        out_pca = pca.fit_transform(X)
        pca_var_pct = 100.0 * float(np.sum(pca.explained_variance_ratio_))
    if n >= 5:
        nn = max(2, min(UMAP_N_NEIGHBORS, n - 1))
        out_umap = UMAP(
            n_components=2,
            n_neighbors=nn,
            min_dist=UMAP_MIN_DIST,
            metric="euclidean",
            random_state=random_state,
            verbose=False,
        ).fit_transform(X)
        tsne_perp = int(max(5, min(TSNE_PERPLEXITY, n - 1)))
        out_tsne = TSNE(
            n_components=2,
            perplexity=tsne_perp,
            learning_rate="auto",
            init="pca",
            random_state=random_state,
            verbose=0,
        ).fit_transform(X)
    return {
        "umap": out_umap,
        "tsne": out_tsne,
        "pca": out_pca,
        "tsne_perplexity": tsne_perp,
        "pca_var_pct": pca_var_pct,
    }


def plot_trajectory_umap(
    wide_roll: pd.DataFrame,
    labels: np.ndarray,
    meta_df: pd.DataFrame,
    random_state: int,
    out_png: Path,
    X_umap_feat: np.ndarray | None = None,
    emb: np.ndarray | None = None,
) -> None:
    """UMAP on scaled rolling features, or on precomputed X_umap_feat (e.g. z-score rows for shape mode)."""
    from umap import UMAP

    _ = meta_df
    if emb is None:
        X = _viz_feature_matrix(wide_roll, X_umap_feat)
        n = X.shape[0]
        if n < 5:
            warnings.warn("样本过少，跳过 UMAP")
            return
        nn = max(2, min(UMAP_N_NEIGHBORS, n - 1))
        reducer = UMAP(
            n_components=2,
            n_neighbors=nn,
            min_dist=UMAP_MIN_DIST,
            metric="euclidean",
            random_state=random_state,
            verbose=False,
        )
        emb = reducer.fit_transform(X)
    else:
        emb = np.asarray(emb, dtype=float)
        if emb.shape[0] != len(labels) or not np.any(np.isfinite(emb)):
            warnings.warn("UMAP 坐标无效，跳过作图")
            return
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in np.sort(np.unique(labels)):
        m = labels == c
        ax.scatter(emb[m, 0], emb[m, 1], s=16, alpha=0.72, label=f"C{int(c)} (n={int(m.sum())})")
    ax.set_title("Trajectory UMAP (visualization features)")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(out_png, dpi=160)
    plt.close(fig)


def plot_trajectory_tsne(
    wide_roll: pd.DataFrame,
    labels: np.ndarray,
    meta_df: pd.DataFrame,
    random_state: int,
    out_png: Path,
    X_umap_feat: np.ndarray | None = None,
    emb: np.ndarray | None = None,
    tsne_perplexity: int | None = None,
) -> None:
    """t-SNE2D embedding on the same visualization feature matrix as UMAP."""
    from sklearn.manifold import TSNE

    _ = meta_df
    if emb is None:
        X = _viz_feature_matrix(wide_roll, X_umap_feat)
        n = X.shape[0]
        if n < 5:
            warnings.warn("样本过少，跳过 t-SNE")
            return
        perp = int(max(5, min(TSNE_PERPLEXITY, n - 1)))
        tsne = TSNE(
            n_components=2,
            perplexity=perp,
            learning_rate="auto",
            init="pca",
            random_state=random_state,
            verbose=0,
        )
        emb = tsne.fit_transform(X)
        tsne_perplexity = perp
    else:
        emb = np.asarray(emb, dtype=float)
        if emb.shape[0] != len(labels) or not np.any(np.isfinite(emb)):
            warnings.warn("t-SNE 坐标无效，跳过作图")
            return
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in np.sort(np.unique(labels)):
        m = labels == c
        ax.scatter(emb[m, 0], emb[m, 1], s=16, alpha=0.72, label=f"C{int(c)} (n={int(m.sum())})")
    if tsne_perplexity is not None:
        ax.set_title(f"Trajectory t-SNE (perplexity={tsne_perplexity}, visualization features)")
    else:
        ax.set_title("Trajectory t-SNE (visualization features)")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(out_png, dpi=160)
    plt.close(fig)


def plot_trajectory_pca(
    wide_roll: pd.DataFrame,
    labels: np.ndarray,
    meta_df: pd.DataFrame,
    out_png: Path,
    X_umap_feat: np.ndarray | None = None,
    emb: np.ndarray | None = None,
    pca_var_pct: float | None = None,
) -> None:
    """PCA 2D projection on the same visualization feature matrix as UMAP."""
    from sklearn.decomposition import PCA

    _ = meta_df
    if emb is None:
        X = _viz_feature_matrix(wide_roll, X_umap_feat)
        n = X.shape[0]
        if n < 3:
            warnings.warn("样本过少，跳过 PCA")
            return
        pca = PCA(n_components=2)
        emb = pca.fit_transform(X)
        pca_var_pct = 100.0 * float(np.sum(pca.explained_variance_ratio_))
    else:
        emb = np.asarray(emb, dtype=float)
        if emb.shape[0] != len(labels) or not np.any(np.isfinite(emb)):
            warnings.warn("PCA 坐标无效，跳过作图")
            return
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in np.sort(np.unique(labels)):
        m = labels == c
        ax.scatter(emb[m, 0], emb[m, 1], s=16, alpha=0.72, label=f"C{int(c)} (n={int(m.sum())})")
    if pca_var_pct is not None:
        ax.set_title(
            f"Trajectory PCA (PC1–PC2, {pca_var_pct:.1f}% variance; visualization features)"
        )
    else:
        ax.set_title("Trajectory PCA (PC1–PC2; visualization features)")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(out_png, dpi=160)
    plt.close(fig)


def merge_traj_embeddings_into_panel(
    panel: pd.DataFrame,
    meta_units: pd.DataFrame,
    emb_pack: dict[str, Any],
    hier_distance_mode: str,
    rolling_window: int,
    trajectory_cluster: np.ndarray | None = None,
) -> pd.DataFrame:
    """
    Left-join trajectory-level 2D coords onto long panel rows by (country, topic).
    Units not in meta_units get NaN embeddings and NaN trajectory_cluster.
    """
    side: dict[str, Any] = {
        "country": meta_units["country"].astype(str).values,
        "topic": meta_units["topic"].values,
        "traj_umap_1": emb_pack["umap"][:, 0],
        "traj_umap_2": emb_pack["umap"][:, 1],
        "traj_tsne_1": emb_pack["tsne"][:, 0],
        "traj_tsne_2": emb_pack["tsne"][:, 1],
        "traj_pca_1": emb_pack["pca"][:, 0],
        "traj_pca_2": emb_pack["pca"][:, 1],
    }
    if trajectory_cluster is not None:
        side["trajectory_cluster"] = np.asarray(trajectory_cluster, dtype=int)
    traj_side = pd.DataFrame(side)
    out = panel.merge(traj_side, on=["country", "topic"], how="left", validate="m:1")
    out["traj_embed_hier_distance_mode"] = hier_distance_mode
    out["traj_embed_rolling_window"] = int(rolling_window)
    return out


def build_trajectory_units_dimred_csv_df(
    meta_units: pd.DataFrame,
    trajectory_cluster: np.ndarray,
    emb_pack: dict[str, Any],
    hier_distance_mode: str,
    rolling_window: int,
) -> pd.DataFrame:
    """One row per (country, topic) unit: cluster id + UMAP / t-SNE / PCA 2D coordinates."""
    df = pd.DataFrame(
        {
            "country": meta_units["country"].astype(str).values,
            "topic": meta_units["topic"].values,
            "trajectory_cluster": np.asarray(trajectory_cluster, dtype=int),
            "umap_1": emb_pack["umap"][:, 0],
            "umap_2": emb_pack["umap"][:, 1],
            "tsne_1": emb_pack["tsne"][:, 0],
            "tsne_2": emb_pack["tsne"][:, 1],
            "pca_1": emb_pack["pca"][:, 0],
            "pca_2": emb_pack["pca"][:, 1],
        }
    )
    df["traj_embed_hier_distance_mode"] = hier_distance_mode
    df["traj_embed_rolling_window"] = int(rolling_window)
    tp = emb_pack.get("tsne_perplexity")
    df["tsne_perplexity_used"] = float(tp) if tp is not None else np.nan
    vpn = emb_pack.get("pca_var_pct")
    df["pca_pc1_pc2_variance_explained_pct"] = float(vpn) if vpn is not None else np.nan
    return df


def score_labels_flat(X_flat: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    """Davies–Bouldin and silhouette on flat Euclidean space (X_score)."""
    import warnings
    uq = np.unique(labels)
    if len(uq) < 2 or len(X_flat) < uq.max() + 2:
        return np.inf, np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        try:
            sil = float(silhouette_score(X_flat, labels, metric="euclidean"))
        except Exception:
            sil = np.nan
        db = float(davies_bouldin_score(X_flat, labels))
    return db, sil


def cluster_size_entropy(labels: np.ndarray) -> float:
    vc = pd.Series(labels).value_counts(normalize=True)
    return float(-(vc * np.log(vc + 1e-15)).sum())


def build_pairwise_distance_matrix(
    wide_roll: pd.DataFrame, distance_mode: str
) -> tuple[np.ndarray | None, np.ndarray]:
    """
    Build distances for hierarchical clustering and the 2D matrix used for scoring / visualization.

    Returns
    -------
    D : ndarray or None
        Square precomputed distance matrix (symmetric, zero diagonal) for dtw/shape; None for euclidean.
    X_used : ndarray
        Feature matrix aligned with distance_mode (column-standardized rolling for euclidean/dtw;
        row z-normalized trajectories for shape).
    """
    X_raw = wide_roll.to_numpy(dtype=float, copy=False)
    if distance_mode == "euclidean":
        X_used = StandardScaler().fit_transform(X_raw)
        return None, X_used
    if distance_mode == "dtw":
        try:
            from tslearn.metrics import cdist_dtw
        except ImportError as e:
            raise ImportError("tslearn is required for HIER_DISTANCE_MODE='dtw'. Install with: uv add tslearn") from e
        X_used = StandardScaler().fit_transform(X_raw)
        X3 = X_raw[:, :, np.newaxis]
        D = np.asarray(cdist_dtw(X3, X3), dtype=float)
        D = (D + D.T) * 0.5
        np.fill_diagonal(D, 0.0)
        D = np.nan_to_num(D, nan=0.0, posinf=0.0, neginf=0.0)
        return D, X_used
    if distance_mode == "shape":
        Z = z_normalize_rows(wide_roll)
        X_used = Z
        # Pearson correlation matrix; distance = 1 - corr
        with np.errstate(invalid="ignore"):
            C = np.corrcoef(Z)
        D = 1.0 - C
        np.fill_diagonal(D, 0.0)
        D = np.nan_to_num(D, nan=1.0, posinf=1.0, neginf=1.0)
        D = (D + D.T) * 0.5
        np.fill_diagonal(D, 0.0)
        D = np.clip(D, 0.0, 2.0)
        return D, X_used
    raise ValueError(distance_mode)


def fit_hierarchical_single_k(
    wide_roll: pd.DataFrame,
    k: int,
    distance_mode: str,
    linkage_method: str,
    D: np.ndarray | None,
    X_used: np.ndarray,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Fit one AgglomerativeClustering with k clusters; labels are contiguous integers from 0."""
    from sklearn.cluster import AgglomerativeClustering

    n = len(wide_roll)
    if k < 2 or k >= n:
        raise ValueError(f"invalid k={k} for n={n}")
    if distance_mode == "euclidean":
        model = AgglomerativeClustering(
            n_clusters=k, metric="euclidean", linkage=linkage_method
        )
        labels = model.fit_predict(X_used)
    else:
        if D is None:
            raise ValueError("D required for dtw/shape")
        model = AgglomerativeClustering(
            n_clusters=k, metric="precomputed", linkage=linkage_method
        )
        labels = model.fit_predict(D)
    labels = np.asarray(labels, dtype=int)
    # Remap to 0..C-1 if sklearn returns unused ids (defensive)
    uniq = np.unique(labels)
    if len(uniq) != len(np.arange(uniq.min(), uniq.max() + 1)):
        labels = pd.factorize(labels, sort=True)[0].astype(int)
    extras: dict[str, Any] = {
        "distance_mode": distance_mode,
        "linkage_method": linkage_method,
        "distance_matrix": D,
    }
    return labels, extras


def hierarchical_search_all_k(
    wide_roll: pd.DataFrame,
    k_list: list[int],
    distance_mode: str,
    linkage_method: str,
) -> tuple[pd.DataFrame, np.ndarray | None, np.ndarray]:
    """One precomputed D per run; metrics rows include k, db, silhouette, cluster_sizes, size_entropy."""
    D, X_used = build_pairwise_distance_matrix(wide_roll, distance_mode)
    X_score = X_used
    n = len(wide_roll)
    rows: list[dict[str, Any]] = []
    for k in k_list:
        if k < 2 or k >= n:
            rows.append({"k": k, "error": "k_out_of_range"})
            continue
        try:
            labels, _ = fit_hierarchical_single_k(
                wide_roll, k, distance_mode, linkage_method, D, X_used
            )
            uq = np.unique(labels)
            if len(uq) < 2:
                raise ValueError("single_cluster")
            db, sil = score_labels_flat(X_score, labels)
            sizes = pd.Series(labels).value_counts().sort_index()
            sz_str = ",".join(f"{int(i)}:{int(sizes.loc[i])}" for i in sizes.index)
            rows.append(
                {
                    "k": k,
                    "db": db,
                    "silhouette": sil,
                    "cluster_sizes": sz_str,
                    "size_entropy": cluster_size_entropy(labels),
                    "distance_mode": distance_mode,
                    "linkage_method": linkage_method,
                }
            )
        except Exception as e:
            rows.append({"k": k, "error": str(e)})
    return pd.DataFrame(rows), D, X_used


def pick_best_k_from_metrics(
    metrics_df: pd.DataFrame, strategy: str, override: int | None, n_samples: int
) -> tuple[int, str]:
    """metrics_lex: min (db, -silhouette); fixed: use override if present in successful search rows."""
    if strategy == "fixed":
        if override is None:
            raise ValueError("FINAL_K_STRATEGY=fixed requires FINAL_K_OVERRIDE")
        kf = int(override)
        if kf < 2 or kf >= n_samples:
            raise ValueError(f"FINAL_K_OVERRIDE out of range: {kf}")
        ok_k = set(metrics_df.loc[metrics_df["db"].notna(), "k"].astype(int))
        if kf not in ok_k:
            raise ValueError(f"k={kf} had no successful fit in metrics_df; adjust TRAJ_K_RANGE or k")
        return kf, "fixed"
    if strategy == "metrics_lex":
        sub = metrics_df.loc[metrics_df["db"].notna()].copy()
        if sub.empty:
            raise RuntimeError("no valid k for metrics_lex")
        sub["_neg_sil"] = -np.nan_to_num(sub["silhouette"].astype(float), nan=-1e9)
        sub = sub.sort_values(["db", "_neg_sil", "k"])
        return int(sub.iloc[0]["k"]), "metrics_lex"
    raise ValueError(strategy)


def plot_hier_k_diagnostics(
    metrics_df: pd.DataFrame, k_final: int | None, out_path: Path, title: str
) -> None:
    sub = metrics_df.loc[metrics_df["db"].notna()].sort_values("k")
    if sub.empty:
        warnings.warn("No valid k rows for diagnostics plot; skipping")
        return
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    ax0, ax1 = axes
    ax0.plot(sub["k"], sub["db"], "o-", color="C1", lw=1.2)
    ax0.set_xlabel("k")
    ax0.set_ylabel("Davies–Bouldin (X_score)")
    ax0.set_title("DB index")
    sil = sub["silhouette"].astype(float)
    ax1.plot(sub["k"], sil, "o-", color="C2", lw=1.2)
    ax1.set_xlabel("k")
    ax1.set_ylabel("Silhouette (X_score)")
    ax1.set_title("Silhouette")
    if k_final is not None:
        for ax in axes:
            ax.axvline(
                k_final,
                color="crimson",
                ls="--",
                lw=1.0,
                alpha=0.85,
                label=f"k_final={k_final}",
            )
            ax.legend(fontsize=7, loc="best")
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_trajectory_dendrogram(
    wide_roll: pd.DataFrame,
    distance_mode: str,
    linkage_method: str,
    D: np.ndarray | None,
    X_used: np.ndarray,
    out_path: Path,
    title: str,
) -> None:
    """Linkage + dendrogram; truncated when many leaves for readability."""
    from scipy.cluster.hierarchy import dendrogram, linkage
    from scipy.spatial.distance import pdist, squareform

    if distance_mode == "euclidean":
        if linkage_method == "ward":
            Z = linkage(X_used, method="ward")
        else:
            condensed = pdist(X_used, metric="euclidean")
            Z = linkage(condensed, method=linkage_method)
    else:
        if D is None:
            warnings.warn("No D for dendrogram; skip")
            return
        condensed = squareform(D, checks=False)
        Z = linkage(condensed, method=linkage_method)

    n = len(wide_roll)
    fig, ax = plt.subplots(figsize=(12, 5))
    if n > 80:
        dendrogram(Z, ax=ax, truncate_mode="lastp", p=50, leaf_rotation=90.0, no_labels=True)
    else:
        dendrogram(Z, ax=ax, leaf_rotation=90.0, no_labels=True)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


# === Cell 1: grid search and k selection ===
_file_suffix = HIER_DISTANCE_MODE
_k_list = [int(k) for k in TRAJ_K_RANGE]
search_df, D_main, X_used_main = hierarchical_search_all_k(
    wide_roll_raw, _k_list, HIER_DISTANCE_MODE, HIER_LINKAGE_METHOD
)
_metrics_path = CATCHUP_ROUND2_DIR / f"trajectory_k_search_metrics_hier_{_file_suffix}.csv"
search_df.to_csv(_metrics_path, index=False)
print(f"Hierarchical k search (saved {_metrics_path.name}):\n", search_df.to_string())

k_final, k_strategy_used = pick_best_k_from_metrics(
    search_df, FINAL_K_STRATEGY, FINAL_K_OVERRIDE, len(wide_roll_raw)
)
print(f"\nk_final={k_final} (strategy={k_strategy_used}, configured={FINAL_K_STRATEGY})")

for _, row in search_df.iterrows():
    if pd.notna(row.get("cluster_sizes")):
        print(f"  k={int(row['k'])} cluster_sizes: {row['cluster_sizes']}")

plot_hier_k_diagnostics(
    search_df,
    k_final,
    CATCHUP_ROUND2_DIR / f"trajectory_k_diagnostics_hier_{_file_suffix}.png",
    f"Hierarchical k diagnostics ({HIER_DISTANCE_MODE}, {HIER_LINKAGE_METHOD})",
)
print("Grid search & k selection done. Run the next cell for final clustering outputs.")

Hierarchical k search (saved trajectory_k_search_metrics_hier_dtw.csv):
     k        db  silhouette                                 cluster_sizes  size_entropy distance_mode linkage_method
0   2  1.186439    0.490937                                    0:42,1:620      0.236341           dtw        average
1   3  0.782282    0.460551                                0:620,1:37,2:5      0.259499           dtw        average
2   4  1.355335    0.391898                          0:121,1:499,2:5,3:37      0.721803           dtw        average
3   5  2.390655    0.363986                      0:44,1:77,2:5,3:37,4:499      0.841612           dtw        average
4   6  2.123863    0.363512                  0:7,1:77,2:37,3:37,4:499,5:5      0.870734           dtw        average
5   7  1.887082    0.357800              0:77,1:499,2:37,3:37,4:6,5:5,6:1      0.875070           dtw        average
6   8  1.924922    0.345295          0:37,1:499,2:9,3:68,4:6,5:5,6:1,7:37      0.917021           dtw       

In [15]:
# === Cell 2: final hierarchical fit, export, and plots ===
k_final = 4
labels_hier, hier_extras = fit_hierarchical_single_k(
    wide_roll_raw,
    k_final,
    HIER_DISTANCE_MODE,
    HIER_LINKAGE_METHOD,
    D_main,
    X_used_main,
)

_out_suffix = HIER_DISTANCE_MODE
pd.DataFrame(
    {"country": meta["country"], "topic": meta["topic"], "trajectory_cluster": labels_hier}
).to_csv(CATCHUP_ROUND2_DIR / f"trajectory_clusters_hier_{_out_suffix}.csv", index=False)

plot_trajectory_prototypes(
    wide_roll_raw,
    labels_hier,
    years_all,
    CATCHUP_ROUND2_DIR / f"trajectory_prototypes_hier_{_out_suffix}.png",
    f"Hierarchical prototypes (k={k_final}, mode={HIER_DISTANCE_MODE})",
)
_umap_X = X_used_main if HIER_DISTANCE_MODE == "shape" else None
_emb_pack = fit_trajectory_2d_embeddings(wide_roll_raw, _umap_X, RANDOM_STATE)
_traj_units_dimred_df = build_trajectory_units_dimred_csv_df(
    meta,
    labels_hier,
    _emb_pack,
    HIER_DISTANCE_MODE,
    MAIN_ROLLING_WINDOW,
)
_traj_units_dimred_csv = (
    CATCHUP_ROUND2_DIR / f"trajectory_units_dimred_2d_hier_{_out_suffix}.csv"
)
_traj_units_dimred_df.to_csv(_traj_units_dimred_csv, index=False)
print("Saved trajectory units + 2D coords (UMAP/t-SNE/PCA):", _traj_units_dimred_csv.resolve())

plot_trajectory_umap(
    wide_roll_raw,
    labels_hier,
    meta,
    RANDOM_STATE,
    CATCHUP_ROUND2_DIR / f"trajectory_umap_hier_{_out_suffix}.png",
    X_umap_feat=_umap_X,
    emb=_emb_pack["umap"],
)
plot_trajectory_tsne(
    wide_roll_raw,
    labels_hier,
    meta,
    RANDOM_STATE,
    CATCHUP_ROUND2_DIR / f"trajectory_tsne_hier_{_out_suffix}.png",
    X_umap_feat=_umap_X,
    emb=_emb_pack["tsne"],
    tsne_perplexity=_emb_pack["tsne_perplexity"],
)
plot_trajectory_pca(
    wide_roll_raw,
    labels_hier,
    meta,
    CATCHUP_ROUND2_DIR / f"trajectory_pca_hier_{_out_suffix}.png",
    X_umap_feat=_umap_X,
    emb=_emb_pack["pca"],
    pca_var_pct=_emb_pack["pca_var_pct"],
)

_panel_with_traj_emb = merge_traj_embeddings_into_panel(
    panel_main,
    meta,
    _emb_pack,
    HIER_DISTANCE_MODE,
    MAIN_ROLLING_WINDOW,
    trajectory_cluster=labels_hier,
)
_panel_emb_csv = (
    CATCHUP_ROUND2_DIR / f"topic_country_year_panel_with_traj_embeddings_hier_{_out_suffix}.csv"
)
_panel_with_traj_emb.to_csv(_panel_emb_csv, index=False)
print("Saved panel + trajectory 2D embeddings:", _panel_emb_csv.resolve())

_mvp_panel_emb_csv = (
    CATCHUP_MVP_DIR / f"topic_country_year_panel_with_traj_embeddings_hier_{_out_suffix}.csv"
)
_panel_with_traj_emb.to_csv(_mvp_panel_emb_csv, index=False)
print("Copy next to MVP topic_country_year_panel.csv:", _mvp_panel_emb_csv.resolve())

plot_trajectory_dendrogram(
    wide_roll_raw,
    HIER_DISTANCE_MODE,
    HIER_LINKAGE_METHOD,
    D_main,
    X_used_main,
    CATCHUP_ROUND2_DIR / f"trajectory_dendrogram_hier_{_out_suffix}.png",
    f"Dendrogram ({HIER_DISTANCE_MODE}, {HIER_LINKAGE_METHOD}, n={len(wide_roll_raw)})",
)

print("Final hierarchical cluster sizes:\n", pd.Series(labels_hier).value_counts().sort_index())

if SWEEP_ROLLING_WINDOWS:
    sweep_rows = []
    for rw in SWEEP_ROLLING_WINDOWS:
        wr, _, _, _ = build_trajectory_matrix(
            panel_main, years_all, rw, ROLLING_MIN_PERIODS, MIN_ACTIVE_YEARS
        )
        if k_final >= len(wr):
            sweep_rows.append({"rolling_window": rw, "error": "n_samples_too_small_for_k"})
            continue
        D_sw, Xu_sw = build_pairwise_distance_matrix(wr, HIER_DISTANCE_MODE)
        lab, _ = fit_hierarchical_single_k(
            wr, k_final, HIER_DISTANCE_MODE, HIER_LINKAGE_METHOD, D_sw, Xu_sw
        )
        X_score_sw = Xu_sw
        db, sil = score_labels_flat(X_score_sw, lab)
        sizes = pd.Series(lab).value_counts().sort_index()
        sz_str = ",".join(f"{int(i)}:{int(sizes.loc[i])}" for i in sizes.index)
        sweep_rows.append(
            {
                "rolling_window": rw,
                "k": k_final,
                "n_samples": len(wr),
                "db": db,
                "silhouette": sil,
                "size_entropy": cluster_size_entropy(lab),
                "cluster_sizes": sz_str,
                "distance_mode": HIER_DISTANCE_MODE,
                "linkage_method": HIER_LINKAGE_METHOD,
            }
        )
    sweep_df = pd.DataFrame(sweep_rows)
    print("\nRolling-window sweep (fixed k_final):\n", sweep_df.to_string())
    sweep_df.to_csv(
        CATCHUP_ROUND2_DIR / f"hier_{HIER_DISTANCE_MODE}_rolling_window_sweep.csv", index=False
    )


/Users/luoyiti/Project/catch-up/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved trajectory units + 2D coords (UMAP/t-SNE/PCA): /Users/luoyiti/Project/catch-up/output/time_series_hierarchical/trajectory_units_dimred_2d_hier_dtw.csv
Saved panel + trajectory 2D embeddings: /Users/luoyiti/Project/catch-up/output/time_series_hierarchical/topic_country_year_panel_with_traj_embeddings_hier_dtw.csv
Copy next to MVP topic_country_year_panel.csv: /Users/luoyiti/Project/catch-up/output/catchup_mvp/topic_country_year_panel_with_traj_embeddings_hier_dtw.csv
Final hierarchical cluster sizes:
 0    121
1    499
2      5
3     37
Name: count, dtype: int64


## 6. 备注

若本 notebook 报错缺少 `paper_topics.csv` 或主数据 CSV，请先在同一环境中运行上游 **KeyBERT 聚类 / 导出** 相关 notebook，使 `config/cluster_keybert_from_cluster.json` 中的路径可用，并生成 `output/catchup_mvp/topic_country_year_panel.csv`（可选，但可加速读取）。

**dtw** 模式若提示缺少 `tslearn`，请执行 `uv sync` 或 `pip install tslearn`。
